# 72. Edit Distance

## Topic Alignment
- Edit distance is fundamental in NLP for spell checking, DNA sequence alignment in bioinformatics, version control diff algorithms, and fuzzy string matching in search engines and recommendation systems.

## Metadata Summary
- Source: https://leetcode.com/problems/edit-distance/
- Tags: Dynamic Programming, String
- Difficulty: Hard
- Priority: High

## Problem Statement
Given two strings `word1` and `word2`, return the minimum number of operations required to convert `word1` to `word2`.

You have the following three operations permitted on a word:

- **Insert** a character
- **Delete** a character
- **Replace** a character

## Progressive Hints
- Hint 1: Use 2D DP where dp[i][j] represents edit distance between word1[0:i] and word2[0:j].
- Hint 2: If characters match, dp[i][j] = dp[i-1][j-1] (no operation needed).
- Hint 3: If characters don't match, consider three operations: insert, delete, replace.
- Hint 4: Insert: dp[i][j-1] + 1, Delete: dp[i-1][j] + 1, Replace: dp[i-1][j-1] + 1.
- Hint 5: Take the minimum of the three operations.
- Hint 6: Base cases: dp[0][j] = j (insert j chars), dp[i][0] = i (delete i chars).

## Solution Overview
Use 2D dynamic programming (also known as **Levenshtein Distance**).

**State Definition**: `dp[i][j]` = minimum edit distance to convert `word1[0:i]` to `word2[0:j]`

**Recurrence**:
```python
if word1[i-1] == word2[j-1]:
    dp[i][j] = dp[i-1][j-1]  # Characters match, no operation
else:
    dp[i][j] = 1 + min(
        dp[i-1][j],      # Delete from word1
        dp[i][j-1],      # Insert to word1 (or delete from word2)
        dp[i-1][j-1]     # Replace
    )
```

**Base cases**:
- `dp[0][j] = j` (insert j characters)
- `dp[i][0] = i` (delete i characters)

## Detailed Explanation

### Understanding the Operations

Given `word1` and `word2`, we're transforming `word1` into `word2`. Each operation has a clear interpretation:

1. **Insert**: Add a character to word1
   - Example: "abc" → "abxc" (insert 'x')

2. **Delete**: Remove a character from word1
   - Example: "abc" → "ac" (delete 'b')

3. **Replace**: Change a character in word1
   - Example: "abc" → "axc" (replace 'b' with 'x')

---

### DP Table Construction

**State**: `dp[i][j]` = edit distance between `word1[0:i]` and `word2[0:j]`

**Base Cases**:
- `dp[0][0] = 0` (both strings empty)
- `dp[i][0] = i` (delete all i characters from word1)
- `dp[0][j] = j` (insert all j characters to empty string)

**Recurrence Logic** (at position i, j):

**Case 1**: Characters match (`word1[i-1] == word2[j-1]`)
- No operation needed, inherit from diagonal
- `dp[i][j] = dp[i-1][j-1]`

**Case 2**: Characters don't match
- **Option A - Delete**: Delete word1[i-1], match word1[0:i-1] with word2[0:j]
  - Cost: `dp[i-1][j] + 1`
  
- **Option B - Insert**: Insert word2[j-1] into word1, match word1[0:i] with word2[0:j-1]
  - Cost: `dp[i][j-1] + 1`
  - Equivalent to: already matched word1[0:i] with word2[0:j-1], now insert word2[j-1]
  
- **Option C - Replace**: Replace word1[i-1] with word2[j-1]
  - Cost: `dp[i-1][j-1] + 1`

- Take minimum: `dp[i][j] = 1 + min(delete, insert, replace)`

---

### Example Walkthrough

**word1 = "horse", word2 = "ros"**

DP Table:
```
      ""  r  o  s
""     0  1  2  3
h      1  1  2  3
o      2  2  1  2
r      3  2  2  2
s      4  3  3  2
e      5  4  4  3
```

**Step-by-step for dp[3][1] ("hor" → "r")**:
- word1[2] = 'r', word2[0] = 'r' → **match!**
- dp[3][1] = dp[2][0] = 2

**Step-by-step for dp[5][3] ("horse" → "ros")**:
- word1[4] = 'e', word2[2] = 's' → **no match**
- Delete 'e': dp[4][3] + 1 = 2 + 1 = 3
- Insert 's': dp[5][2] + 1 = 4 + 1 = 5
- Replace 'e' with 's': dp[4][2] + 1 = 3 + 1 = 4
- Minimum = 3

**Operations**: horse → rorse (replace 'h' with 'r') → rose (delete 'r') → ros (delete 'e')
Or: horse → hors (delete 'e') → ros (replace 'h' with 'r') → ros (delete 'o'...)
Actually optimal: horse → rorse → rose → ros = 3 operations

---

### Space Optimization

Since we only need the previous row to compute the current row, we can optimize to O(min(m, n)) space:

```python
prev = list(range(n + 1))
for i in range(1, m + 1):
    curr = [i]  # First element is dp[i][0] = i
    for j in range(1, n + 1):
        if word1[i-1] == word2[j-1]:
            curr.append(prev[j-1])
        else:
            curr.append(1 + min(prev[j], curr[j-1], prev[j-1]))
    prev = curr
return prev[n]
```

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| 2D DP | O(m×n) | O(m×n) | Standard approach |
| 1D DP | O(m×n) | O(min(m,n)) | Space-optimized |
| Recursion + memo | O(m×n) | O(m×n) | Top-down, clearer logic |
| Hirschberg's algorithm | O(m×n) | O(min(m,n)) | Optimal space, can reconstruct path |

In [ ]:
class Solution:
    def minDistance(self, word1: str, word2: str) -> int:
        """
        2D DP solution for Edit Distance.
        
        Time: O(m×n)
        Space: O(m×n)
        """
        m, n = len(word1), len(word2)
        
        # Create DP table
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        
        # Base cases
        for i in range(m + 1):
            dp[i][0] = i  # Delete all characters from word1
        for j in range(n + 1):
            dp[0][j] = j  # Insert all characters to empty string
        
        # Fill the table
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if word1[i-1] == word2[j-1]:
                    # Characters match, no operation needed
                    dp[i][j] = dp[i-1][j-1]
                else:
                    # Take minimum of three operations
                    dp[i][j] = 1 + min(
                        dp[i-1][j],      # Delete
                        dp[i][j-1],      # Insert
                        dp[i-1][j-1]     # Replace
                    )
        
        return dp[m][n]

In [ ]:
# Test cases
tests = [
    ("horse", "ros", 3),      # Replace h→r, delete r, delete e
    ("intention", "execution", 5),  # Classic example
    ("", "abc", 3),           # Insert all
    ("abc", "", 3),           # Delete all
    ("abc", "abc", 0),        # Already equal
    ("a", "b", 1),            # Single replace
    ("park", "spake", 3),     # Insert s, replace r→a, replace k→e
]

solver = Solution()
for word1, word2, expected in tests:
    result = solver.minDistance(word1, word2)
    assert result == expected, f"Failed for ('{word1}', '{word2}'): got {result}, expected {expected}"
print('All tests passed.')

## Complexity Analysis
- **Time**: O(m×n) where m = len(word1), n = len(word2)
  - Fill m×n table, each cell in O(1)
- **Space**: O(m×n) for 2D table
  - Can optimize to O(min(m,n)) using 1D array

## Edge Cases & Pitfalls
- **Empty strings**: One or both strings empty → return length of non-empty string
- **Identical strings**: Return 0
- **Single character difference**: Should return 1
- **Completely different strings**: Maximum is max(m, n)
- **Index confusion**: Remember dp[i][j] corresponds to word1[i-1] and word2[j-1]
- **Base case initialization**: Ensure row 0 and column 0 are properly initialized
- **Operation semantics**: Insert/delete can be confusing - think carefully about what each means

## Follow-up Variants
- **Weighted operations**: Different costs for insert/delete/replace
- **One edit distance**: Check if strings are exactly one edit apart (LC 161)
- **Delete distance**: Only allow delete operations (related to LCS)
- **Reconstruct edit sequence**: Return the actual operations, not just count
- **Multiple strings**: Edit distance for k strings simultaneously
- **Wildcard matching**: Allow special characters like * and ? (LC 44)
- **Regular expression matching**: Allow . and * patterns (LC 10)

## Takeaways
- Edit Distance (Levenshtein Distance) is one of the most important string DP problems.
- The recurrence elegantly captures the three edit operations.
- Understanding when characters match (inherit from diagonal) vs don't match (take min of three) is crucial.
- This problem pattern extends to many sequence alignment and transformation problems.
- The DP table has clear semantic meaning: dp[i][j] is always the minimum edits for prefixes.
- Edit distance has real-world applications in spell checking, DNA sequencing, and diff algorithms.
- Mastering this problem provides foundation for understanding sequence-to-sequence problems in NLP and bioinformatics.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 1143 | Longest Common Subsequence | Related string DP |
| LC 161 | One Edit Distance | Simplified variant |
| LC 583 | Delete Operation for Two Strings | Edit distance with only deletes |
| LC 712 | Minimum ASCII Delete Sum for Two Strings | Weighted edit distance |
| LC 10 | Regular Expression Matching | More complex string matching |